# Setup

## Library installation

In [3]:
!pip install pytorch_lightning transformers -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 48.2 MB/s eta 0:00:00


## Hugging Face login

In [4]:
import os
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

# Data Handling

## Loading

In [3]:
# Uploading the dataset files
from google.colab import files
uploaded = files.upload()

Saving genia_test_context.json to genia_test_context.json
Saving genia_train_dev_context.json to genia_train_dev_context.json


In [4]:
import json

# Load the training/dev file
with open('genia_train_dev_context.json', 'r') as f:
    train_data = json.load(f)

# Load the test file
with open('genia_test_context.json', 'r') as f:
    test_data = json.load(f)


# Number of train/dev samples
print("Total train samples:", len(train_data))

# Number of test samples
print("Total test samples:", len(test_data))

# Inspecting the structure of the first item
print("Keys:", [f"{k}" for k in train_data[0]])
print(json.dumps(train_data[0], indent=2))

Total train samples: 16692
Total test samples: 1854
Keys: ['tokens', 'entities', 'relations', 'org_id', 'pos', 'ltokens', 'rtokens']
{
  "tokens": [
    "IL-2",
    "gene",
    "expression",
    "and",
    "NF-kappa",
    "B",
    "activation",
    "through",
    "CD28",
    "requires",
    "reactive",
    "oxygen",
    "production",
    "by",
    "5-lipoxygenase",
    "."
  ],
  "entities": [
    {
      "start": 14,
      "end": 15,
      "type": "protein"
    },
    {
      "start": 4,
      "end": 6,
      "type": "protein"
    },
    {
      "start": 0,
      "end": 2,
      "type": "DNA"
    },
    {
      "start": 8,
      "end": 9,
      "type": "protein"
    }
  ],
  "relations": {},
  "org_id": "ge/train/0001",
  "pos": [
    "PROPN",
    "NOUN",
    "NOUN",
    "CCONJ",
    "PROPN",
    "PROPN",
    "NOUN",
    "ADP",
    "PROPN",
    "VERB",
    "ADJ",
    "NOUN",
    "NOUN",
    "ADP",
    "NUM",
    "."
  ],
  "ltokens": [],
  "rtokens": []
}


## Creating torch dataset

In [5]:
import torch
import json
from torch.utils.data import Dataset

class GeniaDataset(Dataset):
    def __init__(self, file_path, tokenizer, max_len=128):
        with open(file_path, 'r') as f:
            self.data = json.load(f)
        self.tokenizer = tokenizer
        self.max_len = max_len

        # Build Label Mapping
        self.label2id = {"O": 0} # "O" is "Outside" (not an entity)
        self.id2label = {0: "O"}
        unique_labels = set()
        for item in self.data:
            for ent in item['entities']:
                unique_labels.add(ent['type'])

        # Assign IDs to labels
        for i, label in enumerate(sorted(unique_labels), start=1):
            self.label2id[label] = i
            self.id2label[i] = label

        print(f"Found {len(unique_labels)} unique labels: {unique_labels}")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        text_tokens = item['tokens'] # Original words
        entities = item['entities']  # Original entities

        # Tokenize with Offset Mapping
        encoding = self.tokenizer(
            text_tokens,
            is_split_into_words=True,
            return_offsets_mapping=True,
            padding='max_length',
            truncation=True,
            max_length=self.max_len,
            return_tensors='pt'
        )

        # Get the map from BERT tokens to original words
        word_ids = encoding.word_ids()

        # Align Entities to BERT Tokens
        valid_spans = []
        valid_labels = []

        for ent in entities:
            word_start = ent['start']
            word_end = ent['end'] - 1 # Convert exclusive to inclusive for mapping
            label_str = ent['type']

            # Find the start token index
            token_start_index = -1
            token_end_index = -1

            # Scan tokens to find the match
            for i, wid in enumerate(word_ids):
                if wid == word_start and token_start_index == -1:
                    token_start_index = i
                if wid == word_end:
                    token_end_index = i # Keep updating until we hit the next word

            # If the entity was truncated by max_len, skip it
            if token_start_index != -1 and token_end_index != -1:
                valid_spans.append([token_start_index, token_end_index])
                valid_labels.append(self.label2id[label_str])

        # Return Tensors
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'spans': torch.tensor(valid_spans, dtype=torch.long),
            'labels': torch.tensor(valid_labels, dtype=torch.long)
        }


## Collate function

In [5]:
import torch

def grid_ner_collate_fn(batch):
    # Stack the Standard BERT Inputs
    input_ids = torch.stack([item['input_ids'] for item in batch])
    attention_mask = torch.stack([item['attention_mask'] for item in batch])

    # Initialize the Empty Answer Grid
    # Shape: (Batch_Size, Max_Len, Max_Len)
    # We fill it with 0 because 0 is the ID for "O" (Outside/No Entity).
    batch_size = len(batch)
    max_len = input_ids.shape[1]
    grid_labels = torch.zeros((batch_size, max_len, max_len), dtype=torch.long)

    # Fill in the Active Cells
    # where an entity exists.
    for i, item in enumerate(batch):
        spans = item['spans']   # Tensor of [start, end] coords
        labels = item['labels'] # Tensor of label IDs

        # Zip them together to iterate
        for (start, end), label in zip(spans, labels):
            # Safety check: ensure coordinates are within the 128 limit
            if start < max_len and end < max_len:
                # Set the cell at [Batch_Index, Start_Index, End_Index] to the Label ID
                grid_labels[i, start, end] = label

    return {
        'input_ids': input_ids,
        'attention_mask': attention_mask,
        'grid_labels': grid_labels
    }

# Model

In [6]:
import torch
import torch.nn as nn
import pytorch_lightning as pl
from transformers import BertModel
from torch.optim import AdamW

class NestedNER(pl.LightningModule):
    def __init__(self, num_labels, id2label=None, learning_rate=2e-5):
        super().__init__()
        self.save_hyperparameters()
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        self.classifier = nn.Sequential(
            nn.Linear(768 * 2, 512),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(512, num_labels)
        )
        self.criterion = nn.CrossEntropyLoss(ignore_index=-100)
        self.validation_step_outputs = []
        self.test_step_outputs = []
        self.id2label = id2label

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = outputs.last_hidden_state
        batch_size, seq_len, hidden_size = sequence_output.shape
        start_rep = sequence_output.unsqueeze(2).expand(-1, -1, seq_len, -1)
        end_rep = sequence_output.unsqueeze(1).expand(-1, seq_len, -1, -1)
        span_grid = torch.cat([start_rep, end_rep], dim=-1)
        logits = self.classifier(span_grid)
        return logits.permute(0, 3, 1, 2)

    def training_step(self, batch, batch_idx):
        input_ids = batch['input_ids']
        mask = batch['attention_mask']
        grid_labels = batch['grid_labels']
        logits = self(input_ids, mask)
        seq_len = input_ids.shape[1]
        triu_mask = torch.triu(torch.ones(seq_len, seq_len, device=self.device)).bool()
        active_targets = grid_labels.clone()
        active_targets[:, ~triu_mask] = -100
        loss = self.criterion(logits, active_targets)
        self.log('train_loss', loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        input_ids = batch['input_ids']
        mask = batch['attention_mask']
        grid_labels = batch['grid_labels']
        logits = self(input_ids, mask)
        seq_len = input_ids.shape[1]
        triu_mask = torch.triu(torch.ones(seq_len, seq_len, device=self.device)).bool()
        active_targets = grid_labels.clone()
        active_targets[:, ~triu_mask] = -100
        loss = self.criterion(logits, active_targets)
        preds = torch.argmax(logits, dim=1)
        target_entities = (active_targets != -100) & (active_targets != 0)
        predicted_entities = (preds != 0) & (active_targets != -100)
        tp = ((preds == active_targets) & target_entities).sum().item()
        fp = ((preds != active_targets) & predicted_entities).sum().item()
        fn = ((preds != active_targets) & target_entities).sum().item()
        self.validation_step_outputs.append({'tp': tp, 'fp': fp, 'fn': fn})
        self.log('val_loss', loss, prog_bar=True)
        return loss

    def on_validation_epoch_end(self):
        outputs = self.validation_step_outputs
        total_tp = sum(x['tp'] for x in outputs)
        total_fp = sum(x['fp'] for x in outputs)
        total_fn = sum(x['fn'] for x in outputs)
        precision = total_tp / (total_tp + total_fp + 1e-9)
        recall = total_tp / (total_tp + total_fn + 1e-9)
        f1 = 2 * (precision * recall) / (precision + recall + 1e-9)
        self.log('val_precision', precision)
        self.log('val_recall', recall)
        self.log('val_f1', f1, prog_bar=True)
        self.validation_step_outputs.clear()

    def test_step(self, batch, batch_idx):
        input_ids = batch['input_ids']
        mask = batch['attention_mask']
        grid_labels = batch['grid_labels']
        logits = self(input_ids, mask)
        seq_len = input_ids.shape[1]
        triu_mask = torch.triu(torch.ones(seq_len, seq_len, device=self.device)).bool()
        active_targets = grid_labels.clone()
        active_targets[:, ~triu_mask] = -100
        loss = self.criterion(logits, active_targets)
        preds = torch.argmax(logits, dim=1)
        target_entities = (active_targets != -100) & (active_targets != 0)
        predicted_entities = (preds != 0) & (active_targets != -100)
        tp = ((preds == active_targets) & target_entities).sum().item()
        fp = ((preds != active_targets) & predicted_entities).sum().item()
        fn = ((preds != active_targets) & target_entities).sum().item()
        self.test_step_outputs.append({'tp': tp, 'fp': fp, 'fn': fn})
        self.log('test_loss', loss, prog_bar=True)
        return loss

    def on_test_epoch_end(self):
        outputs = self.test_step_outputs
        total_tp = sum(x['tp'] for x in outputs)
        total_fp = sum(x['fp'] for x in outputs)
        total_fn = sum(x['fn'] for x in outputs)
        precision = total_tp / (total_tp + total_fp + 1e-9)
        recall = total_tp / (total_tp + total_fn + 1e-9)
        f1 = 2 * (precision * recall) / (precision + recall + 1e-9)
        self.log('test_precision', precision)
        self.log('test_recall', recall)
        self.log('test_f1', f1, prog_bar=True)
        self.test_step_outputs.clear()

    def decode_spans(self, logits, threshold=-1.5):
        batch_size, seq_len, _ = logits.shape
        all_spans = []

        for b in range(batch_size):
            b_logits = logits[b]
            non_o_logits = b_logits[1:, :, :]
            max_non_o_logits, max_non_o_indices = torch.max(non_o_logits, dim=0)
            o_logits = b_logits[0, :, :]

            spans = []
            for start in range(seq_len):
                for end in range(start, seq_len):
                    if (max_non_o_logits[start, end] - o_logits[start, end]) > threshold:
                        label_id = max_non_o_indices[start, end].item() + 1
                        spans.append((start, end, label_id))
            all_spans.append(spans)

        return all_spans

    def configure_optimizers(self):
        return AdamW(self.parameters(), lr=self.hparams.learning_rate)

# Training

In [8]:
from transformers import BertTokenizer
from torch.utils.data import DataLoader
import pytorch_lightning as pl

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

train_dataset = GeniaDataset('genia_train_dev_context.json', tokenizer)
val_dataset = GeniaDataset('genia_test_context.json', tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, collate_fn=grid_ner_collate_fn)
val_loader = DataLoader(val_dataset, batch_size=16, collate_fn=grid_ner_collate_fn)

model = NestedNER(num_labels=len(train_dataset.label2id), id2label=train_dataset.id2label)
trainer = pl.Trainer(max_epochs=5, accelerator='gpu', devices=1)

trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Found 5 unique labels: {'RNA', 'DNA', 'protein', 'cell_type', 'cell_line'}
Found 5 unique labels: {'RNA', 'DNA', 'protein', 'cell_type', 'cell_line'}


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

...

Trainable params: 110 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 110 M                                                                                                
Total estimated model params size (MB): 441                                                                        
Modules in train mode: 6                                                                                           
Modules in eval mode: 228                                                                                          
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/loops/fit_loop.py:534: Found 228 module(s) in eval mode 
at the start of training. This may lead to unexpected behavior during training. If this is intentional, you can 
ignore this warning.

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=5` reached.


In [9]:
repo_id = "epikadith/nest-ner-bert-genia"
model.bert.push_to_hub(repo_id)
tokenizer.push_to_hub(repo_id)

torch.save(model.state_dict(), "model_state.bin")
from huggingface_hub import HfApi
api = HfApi()
api.upload_file(
    path_or_fileobj="model_state.bin",
    path_in_repo="model_state.bin",
    repo_id=repo_id
)

import json

with open("id2label.json", "w") as f:
    json.dump(model.id2label, f)

api.upload_file(
    path_or_fileobj="id2label.json",
    path_in_repo="id2label.json",
    repo_id=repo_id
)

README.md: 0.00B [00:00, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...wqmcaz4/model.safetensors:   0%|          | 17.4kB /  438MB            

No files have been modified since last commit. Skipping to prevent empty commit.


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  model_state.bin             :   9%|9         | 41.9MB /  441MB            

CommitInfo(commit_url='https://huggingface.co/epikadith/nest-ner-bert-genia/commit/f6c7d3d4594640c57334b023300919b2059a59ae', commit_message='Upload id2label.json with huggingface_hub', commit_description='', oid='f6c7d3d4594640c57334b023300919b2059a59ae', pr_url=None, repo_url=RepoUrl('https://huggingface.co/epikadith/nest-ner-bert-genia', endpoint='https://huggingface.co', repo_type='model', repo_id='epikadith/nest-ner-bert-genia'), pr_revision=None, pr_num=None)

# Testing

In [10]:
trainer.test(model, dataloaders=val_loader)

INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│          test_f1          │    0.7240378260612488     │
│         test_loss         │   0.000562600267585367    │
│      test_precision       │    0.7461879849433899     │
│        test_recall        │     0.70316481590271      │
└───────────────────────────┴───────────────────────────┘

[{'test_loss': 0.000562600267585367,
  'test_precision': 0.7461879849433899,
  'test_recall': 0.70316481590271,
  'test_f1': 0.7240378260612488}]

# Inference

In [7]:
import json
import torch
from transformers import BertTokenizer
from huggingface_hub import hf_hub_download

repo_id = "epikadith/nest-ner-bert-genia"

state_dict_path = hf_hub_download(repo_id=repo_id, filename="model_state.bin")
id2label_path = hf_hub_download(repo_id=repo_id, filename="id2label.json")

with open(id2label_path, "r") as f:
    id2label_str = json.load(f)
    loaded_id2label = {int(k): v for k, v in id2label_str.items()}

tokenizer = BertTokenizer.from_pretrained(repo_id)

model = NestedNER(
    num_labels=len(loaded_id2label),
    id2label=loaded_id2label
)
model.load_state_dict(torch.load(state_dict_path, map_location=torch.device('cpu')))
model.eval()

def predict_nested_ner_tuned(text, model, tokenizer, threshold=-2.0):
    model.eval()

    encoding = tokenizer(
        text,
        return_offsets_mapping=True,
        padding='max_length',
        truncation=True,
        max_length=128,
        return_tensors='pt'
    )

    input_ids = encoding['input_ids'].to(model.device)
    attention_mask = encoding['attention_mask'].to(model.device)
    offsets = encoding['offset_mapping'][0].tolist()

    with torch.no_grad():
        logits = model(input_ids, attention_mask)

    logits = logits[0]
    non_o_logits = logits[1:, :, :]
    max_non_o_logits, max_non_o_indices = torch.max(non_o_logits, dim=0)
    o_logits = logits[0, :, :]

    spans = []
    seq_len = input_ids.shape[1]
    for start in range(seq_len):
        for end in range(start, seq_len):
            if (max_non_o_logits[start, end] - o_logits[start, end]) > threshold:
                label_id = max_non_o_indices[start, end].item() + 1
                label = model.id2label[label_id] if model.id2label else label_id
                spans.append((start, end, label))

    results = []
    for start_idx, end_idx, label in spans:
        if start_idx >= len(offsets) or end_idx >= len(offsets):
            continue

        start_char = offsets[start_idx][0]
        end_char = offsets[end_idx][1]

        if start_char == 0 and end_char == 0:
            continue

        entity_text = text[start_char:end_char]
        results.append({"entity": entity_text, "label": label})

    return results

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
sample_text = "The CD28 surface receptor interacts with the NF-kappa B complex."
predictions = predict_nested_ner_tuned(sample_text, model, tokenizer, threshold=-1.5)

for p in predictions:
    print(f"- [{p['label']}] : {p['entity']}")

- [protein] : CD28
- [protein] : CD28 surface receptor
- [protein] : NF-kappa B
- [protein] : NF-kappa B complex


In [13]:
test_sentences = [
    "We evaluated CIITA mRNA levels in human T cells.",
    "The glucocorticoid receptor gene promoter was analyzed.",
    "The CD28 surface receptor interacts with the NF-kappa B complex.",
    "Expression of the human IL-2 gene is activated by the T cell receptor.",
    "EBV-transformed B cell lines were used to study the CD40 ligand."
]

for sen in test_sentences:
  print(f"Sentence: {sen}")
  pred = predict_nested_ner_tuned(sen, model, tokenizer, threshold=-1.5)
  for p in pred:
    print(f"- [{p['label']}] : {p['entity']}")

Sentence: We evaluated CIITA mRNA levels in human T cells.
- [protein] : CIITA
- [RNA] : CIITA mRNA
- [RNA] : mRNA
- [cell_type] : human T cells
- [cell_type] : T cells
Sentence: The glucocorticoid receptor gene promoter was analyzed.
- [protein] : glucocorticoid receptor
- [DNA] : glucocorticoid receptor gene
- [DNA] : glucocorticoid receptor gene promoter
Sentence: The CD28 surface receptor interacts with the NF-kappa B complex.
- [protein] : CD28
- [protein] : CD28 surface receptor
- [protein] : NF-kappa B
- [protein] : NF-kappa B complex
Sentence: Expression of the human IL-2 gene is activated by the T cell receptor.
- [DNA] : human IL-2 gene
- [protein] : IL-2
- [DNA] : IL-2 gene
- [protein] : T cell receptor
Sentence: EBV-transformed B cell lines were used to study the CD40 ligand.
- [cell_line] : EBV-transformed B cell lines
- [protein] : CD40
- [protein] : CD40 ligand
